In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
pyspark.__version__

'3.4.4'

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/05 02:48:32 WARN Utils: Your hostname, LAPTOP-5QMII800 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/05 02:48:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/05 02:48:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-04 03:20:44--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.97, 52.85.39.153, 52.85.39.65, ...
connected. to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.97|:443... 
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet.3’

yellow_tripdata_202 100%[===================>]  61.36M  11.4MB/s    in 7.0s    

2025-03-04 03:20:52 (8.75 MB/s) - ‘yellow_tripdata_2024-10.parquet.3’ saved [64346071/64346071]



In [5]:
output_path = 'data/pq/homework/yellow/2024/10/'

df_yellow = spark.read \
        .parquet('yellow_tripdata_2024-10.parquet')

df_yellow \
        .repartition(4) \
        .write.parquet(output_path, mode='overwrite')

In [59]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [7]:
ls -lh data/pq/homework/yellow/2024/10/

total 90M
-rwxrwxrwx 1 poonk poonk   0 Mar  4 03:21 _SUCCESS*
-rwxrwxrwx 1 poonk poonk 23M Mar  4 03:21 part-00000-cffd1a2c-0ac0-4887-9160-d36906276100-c000.snappy.parquet*
-rwxrwxrwx 1 poonk poonk 23M Mar  4 03:21 part-00001-cffd1a2c-0ac0-4887-9160-d36906276100-c000.snappy.parquet*
-rwxrwxrwx 1 poonk poonk 23M Mar  4 03:21 part-00002-cffd1a2c-0ac0-4887-9160-d36906276100-c000.snappy.parquet*
-rwxrwxrwx 1 poonk poonk 23M Mar  4 03:21 part-00003-cffd1a2c-0ac0-4887-9160-d36906276100-c000.snappy.parquet*


In [8]:
df_yellow.registerTempTable('yellow_trips')

/mnt/c/Users/poonk/spark/spark-3.4.4-bin-hadoop3/python/pyspark/sql/dataframe.py:330: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [13]:
df_day_trips = spark.sql("""
SELECT 
    DATE(tpep_pickup_datetime) as pickup_date,
    COUNT(*) AS trips_count
FROM
    yellow_trips
WHERE
    tpep_pickup_datetime BETWEEN '2024-10-15 00:00:00' AND '2024-10-15 23:59:59'
GROUP BY
   1
""")

In [14]:
df_day_trips.show()

+-----------+-----------+
|pickup_date|trips_count|
+-----------+-----------+
| 2024-10-15|     128893|
+-----------+-----------+



In [41]:
df_longest_trip = spark.sql("""
SELECT
    VendorID,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime) AS hours_diff
FROM
    yellow_trips
ORDER BY
   timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime) DESC
LIMIT 3
""")

In [42]:
df_longest_trip.show()

[Stage 20:====================================>                     (5 + 3) / 8]

+--------+--------------------+---------------------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|hours_diff|
+--------+--------------------+---------------------+----------+
|       2| 2024-10-16 13:03:49|  2024-10-23 07:40:53|       162|
|       2| 2024-10-03 18:47:25|  2024-10-09 18:06:55|       143|
|       2| 2024-10-22 16:00:55|  2024-10-28 09:46:33|       137|
+--------+--------------------+---------------------+----------+



In [29]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-04 03:36:26--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.117, 52.85.39.97, 52.85.39.153, ...
connected. to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.117|:443... 
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-04 03:36:27 (200 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [36]:
df_zones = spark.read \
        .option("header", "true") \
        .csv('taxi_zone_lookup.csv')

In [37]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [39]:
df_result = df_yellow.join(df_zones, df_yellow.PULocationID == df_zones.LocationID)

In [45]:
df_result.registerTempTable('df_least_zone')

/mnt/c/Users/poonk/spark/spark-3.4.4-bin-hadoop3/python/pyspark/sql/dataframe.py:330: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [57]:
df_least_zone = spark.sql("""
SELECT
    PULocationID,
    Zone,
    COUNT(1) as frequency
FROM
    df_least_zone
GROUP BY
   1, 2
ORDER BY
   3 ASC
LIMIT 3
""")

In [58]:
df_least_zone.show()

+------------+--------------------+---------+
|PULocationID|                Zone|frequency|
+------------+--------------------+---------+
|         105|Governor's Island...|        1|
|           5|       Arden Heights|        2|
|         199|       Rikers Island|        2|
+------------+--------------------+---------+

